# research5 v2 — Colab Pro GPU 全自动深入实验套件（无人值守版）

本 Notebook 专为 **Colab Pro 高配 GPU（推荐 `A100` 或 `L4`，预计仅消耗 ~4–10 个 Compute Units）** 设计，完全保留 `v1` 既有成果（零覆盖），并执行 `v2` 全套修复与扩展实验：

1. **Phase 1 — Stage B v2 (`SC400–SC419`, `n=20`)**：加密校验并复用 `v1` 前 10 人（`SC400–SC409`），完成新增 10 人（`SC410–SC419`）整夜高精度评估，检验独立内部复现与 `train/valid/test` 分层。
2. **Phase 2 — 修复版 Stage C v2（整夜无裁剪 `2×2` Oracle + 无泄漏 8 组基线, `n=20`)**：彻底消除 `v1` 的 `112-epoch` 窗边界拼接伪影与干净夜泄漏，对比动态丢弃损坏通道的 `blind_channel_reroute`、因果在线滑动归一化 `causal_running_norm`、`clip_winsor_5iqr`、质量掩码与整段分段重启。
3. **Phase 3 — 修复并升级 Stage C 小模型 7 组对比（5 折 CV × 3 种子 × 31 维严格单 Epoch 特征）**：测试已知干扰（`flat/clip/noise`）及**未知分布外 OOD 干扰（60Hz 工频 / EMG 肌电爆发 / 慢漂移突变）**。
4. **Phase 4 — 修复版 CAP 临床外部复核（`n1, n2, n3`, 原生 `512Hz / 256Hz` 干预）**：整夜无裁剪 `2×2`、YASA 对比、多采样率物理盲检及原生 `512Hz` 无混叠 `60Hz+120Hz` 谐波验证。
5. **Phase 5 — `3A–3E` 敏感性与机制压力套件（`n=20`）**：`3A` 8h纯睡眠窗口 vs 24h、`3B` 时程 (`10–300s`) 与噪声梯度 (`25–500 µV × 3 种子`) + 真实伪迹召回率、`3C` 多通道组合与 13 层 SkipAttention `Δα` 图谱、`3D` `run1/run2/run3/ensemble3` 多检查点稳定度、`3E` `-20..+20` Epoch 冲激滞后曲线与基线置信边际/过渡期脆弱性分层。

---
### 挂机运行步骤（可放心关闭本地电脑）
1. 菜单栏点击 **Runtime（代码执行程序）→ Change runtime type（更改运行时类型）**，硬件加速器选 **`A100 GPU`**（最快约 15–25 分钟）或 **`L4 GPU`**（约 35–55 分钟）。
2. 若需关机离席，请在 Colab Pro 菜单确认开启 **Background execution（后台执行）**。
3. 点击 **Runtime → Run all（全部运行）**，并在弹出窗口中允许挂载 Google Drive。
4. 看到下方开始打印 `[StageB-v2]` / `Phase` 日志后即可关闭网页和本地电脑。所有数据、断点、`6` 张论文级高清图、[`FINAL_AUTONOMOUS_REPORT_V2.md`](file:///content/drive/MyDrive/research5/FINAL_AUTONOMOUS_REPORT_V2.md) 与完整导出包 `exports/research5-autonomous-results-v2.zip` 都会在每个受试者/阶段完成时实时写入 `Google Drive / MyDrive / research5`。


In [ ]:
#@title 1. 挂载 Google Drive、校验 GPU 并全程前台无人值守运行 Phase 0 -> Phase 5
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# ---- 运行参数配置 ----
REQUIRE_GPU = True       # 若仅在 CPU 上调试可改为 False；科研正式跑建议保持 True
SMALL_MODEL_EPOCHS = 75  # 7 arms x 5 folds x 3 seeds 训练轮数

# 1) 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

project = Path('/content/drive/MyDrive/research5')
assert project.is_dir(), f'未找到 Google Drive 目录: {project}'

# 2) 校验 GPU 资源
print('=== GPU Hardware Check (nvidia-smi) ===', flush=True)
smi = subprocess.run(['nvidia-smi'], text=True)
if REQUIRE_GPU and smi.returncode != 0:
    raise RuntimeError('当前不是 GPU 运行时！请在 Colab 菜单点击 Runtime -> Change runtime type -> 选择 A100 或 L4 GPU。')

# 3) 校验基础代码包与 v2 扩展脚本
bundle = project / 'bundles/research5-colab-drive.zip'
assert bundle.is_file(), f'缺少基础代码包: {bundle}'
drv_followup = project / 'followup'
assert (drv_followup / 'autonomous_drive_chain_v2.py').is_file(), f'缺少 v2 主控脚本: {drv_followup / "autonomous_drive_chain_v2.py"}'

# 4) 安全解压基础代码并同步最新 *_v2.py 及 20 人 Manifest 至本地高速运行区 /content/research5-exec/v2
exec_root = Path('/content/research5-exec/v2')
exec_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle, 'r') as zf:
    for item in zf.infolist():
        target = (exec_root / item.filename).resolve()
        assert target.is_relative_to(exec_root.resolve()), f'非法压缩包路径: {item.filename}'
    zf.extractall(exec_root)

(exec_root / 'followup').mkdir(parents=True, exist_ok=True)
copied_v2 = []
for py_file in sorted(drv_followup.glob('*_v2.py')):
    shutil.copyfile(py_file, exec_root / 'followup' / py_file.name)
    copied_v2.append(py_file.name)
print('Synced v2 modules:', copied_v2, flush=True)

manifest_20 = project / 'manifests/development_manifest_20.json'
if manifest_20.is_file():
    (exec_root / 'manifests').mkdir(parents=True, exist_ok=True)
    shutil.copyfile(manifest_20, exec_root / 'manifests/development_manifest_20.json')

assert (exec_root / 'vendor/anysleep/examples/anysleep_no_hydra.py').is_file(), '缺少 AnySleep vendor 文件'
assert (exec_root / 'pilot/support.py').is_file(), '缺少 pilot/support.py'

print('\n======================================================================')
print('驱动检查完成，即将启动无人值守 GPU 实验链。')
print('确认开启 Colab Background execution（后台执行）后，您现在即可安全关闭电脑！')
print('======================================================================\n', flush=True)

# 5) 前台阻塞执行（实时流式输出并按受试者/阶段自动落盘至 Google Drive）
cmd = [
    sys.executable,
    '-u',
    str(exec_root / 'followup/autonomous_drive_chain_v2.py'),
    '--project-root',
    str(project),
    '--exec-root',
    str(exec_root),
    '--device',
    'auto',
    '--small-model-epochs',
    str(SMALL_MODEL_EPOCHS),
]
if REQUIRE_GPU:
    cmd.append('--require-gpu')

subprocess.run(cmd, check=True)
os.sync()
print('\n[DONE] 全部 v2 GPU 实验、高清图与报告压缩包均已持久化保存至 Google Drive！', flush=True)


In [ ]:
#@title 2. 查看运行状态、最新报告与生成的高清图表（随时可单独运行）
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

project = Path('/content/drive/MyDrive/research5')
status_path = project / 'status/autonomous-pipeline-v2/status.json'
if status_path.is_file():
    st = json.loads(status_path.read_text(encoding='utf-8'))
    print('=== Pipeline Status ===')
    print(json.dumps({
        'status': st.get('status'),
        'phase': st.get('phase'),
        'updated_utc': st.get('updated_utc'),
        'completed_phases': st.get('completed_phases', []),
    }, indent=2, ensure_ascii=False))
else:
    print('尚未生成 status.json')

fig_dir = project / 'reports/autonomous-v2/figures'
if fig_dir.is_dir():
    for png in sorted(fig_dir.glob('*.png')):
        print('Figure:', png.name)
        display(Image(filename=str(png), width=900))

rep_path = project / 'FINAL_AUTONOMOUS_REPORT_V2.md'
if rep_path.is_file():
    display(Markdown(rep_path.read_text(encoding='utf-8')))
zip_path = project / 'exports/research5-autonomous-results-v2.zip'
if zip_path.is_file():
    print(f'Export Zip ready: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)')
